# Accessibility Repo Tracker

The GitHub API exposes open issues, pull requests, and release notes for any public repository. This notebook wraps it for accessibility-related repos — axe-core, eslint-plugin-jsx-a11y, and NVDA — so you can ask "are there any open axe-core issues related to combobox keyboard handling?" or "what changed in the latest eslint-plugin-jsx-a11y release?" without leaving your workflow. Directly useful for a daily PR-review habit.

## How it works

The model is given two tools:
- `search_repo_issues(repo, query)` — searches open issues/PRs in a repo for a keyword.
- `get_latest_release(repo)` — fetches the latest release notes for a repo.

The model picks the right tool (and `repo`) based on the question, and can call either one — even multiple times in one turn — across a conversation as you ask about different repos.

In [1]:
# imports

import os
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
github_token = os.getenv('GITHUB_TOKEN')

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:8]}")
else:
    print("Anthropic API Key not set")

if github_token:
    print(f"GitHub token exists and begins {github_token[:4]}")
else:
    print("GitHub token not set — requests will be rate-limited to 60/hour without one")

openai = OpenAI(
    api_key=anthropic_api_key,
    base_url="https://api.anthropic.com/v1/",
)
MODEL = "claude-sonnet-4-6"

GITHUB_HEADERS = {"Accept": "application/vnd.github+json"}
if github_token:
    GITHUB_HEADERS["Authorization"] = f"Bearer {github_token}"

ACCESSIBILITY_REPOS = {
    "axe-core": "dequelabs/axe-core",
    "eslint-plugin-jsx-a11y": "jsx-eslint/eslint-plugin-jsx-a11y",
    "nvda": "nvaccess/nvda",
}

Anthropic API Key exists and begins sk-ant-a
GitHub token exists and begins ghp_


## Step 1: GitHub tool functions and tool schemas

Defines `resolve_repo()` to map friendly names to `owner/name` slugs, `search_repo_issues()` to return up to 5 matching open issues, `get_latest_release()` to fetch the latest release metadata and notes, and the `tools` list with OpenAI-compatible function schemas for both.

In [3]:
def resolve_repo(repo):
    return ACCESSIBILITY_REPOS.get(repo.lower(), repo)

def search_repo_issues(repo, query):
    full_repo = resolve_repo(repo)
    search_url = "https://api.github.com/search/issues"
    params = {"q": f"repo:{full_repo} is:issue is:open {query}"}
    response = requests.get(search_url, headers=GITHUB_HEADERS, params=params).json()
    items = response.get("items", [])[:5]
    return [{"title": i["title"], "number": i["number"], "url": i["html_url"],
             "labels": [l["name"] for l in i.get("labels", [])], "created_at": i["created_at"]} for i in items]

def get_latest_release(repo):
    full_repo = resolve_repo(repo)
    url = f"https://api.github.com/repos/{full_repo}/releases/latest"
    r = requests.get(url, headers=GITHUB_HEADERS).json()
    return {"tag": r.get("tag_name"), "name": r.get("name"), "url": r.get("html_url"),
            "published_at": r.get("published_at"), "notes": (r.get("body") or "")[:1500]}

search_issues_function = {
    "name": "search_repo_issues",
    "description": "Search open issues and pull requests in an accessibility-related GitHub repo by keyword.",
    "parameters": {
        "type": "object",
        "properties": {
            "repo": {"type": "string", "description": "Repo name: 'axe-core', 'eslint-plugin-jsx-a11y', 'nvda', or any 'owner/name'."},
            "query": {"type": "string", "description": "Keyword(s) to search for in issue titles and bodies, e.g. 'combobox keyboard'."},
        },
        "required": ["repo", "query"],
        "additionalProperties": False,
    },
}

latest_release_function = {
    "name": "get_latest_release",
    "description": "Get the latest release notes for an accessibility-related GitHub repo.",
    "parameters": {
        "type": "object",
        "properties": {
            "repo": {"type": "string", "description": "Repo name: 'axe-core', 'eslint-plugin-jsx-a11y', 'nvda', or any 'owner/name'."},
        },
        "required": ["repo"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": search_issues_function}, {"type": "function", "function": latest_release_function}]

## Step 2: System prompt and tool dispatch

Defines the `system_prompt` that configures the assistant as an accessibility GitHub tracker, and `handle_tool_call()` which iterates over `message.tool_calls`, dispatches each by name to the matching Python function, and returns a list of tool response messages.

In [4]:
system_prompt = """You are an accessibility engineer's daily assistant, with live access to GitHub issues, pull \
requests, and release notes for axe-core, eslint-plugin-jsx-a11y, and NVDA.

Call search_repo_issues when the developer asks about open issues, bugs, or pull requests on a topic — \
e.g. "are there any open axe-core issues related to combobox keyboard handling?"

Call get_latest_release when the developer asks what changed in a recent release — e.g. "what changed in \
the latest eslint-plugin-jsx-a11y release?"

When summarizing issues, list each one with its number, title, and link, and group by label if labels are \
informative. When summarizing release notes, pull out the highlights relevant to accessibility rather than \
pasting the entire changelog verbatim.

If a search returns no results, say so plainly."""

def handle_tool_call(message):
    results = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        if tool_call.function.name == "search_repo_issues":
            result = search_repo_issues(**arguments)
        elif tool_call.function.name == "get_latest_release":
            result = get_latest_release(**arguments)
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id,
        })
    return results

## Step 3: Chat function with tool calling

Defines `conversation_history` and `chat()`, which appends the user message, calls the model with `tools` enabled, handles any tool calls by extending the message list with their results, then returns the final assistant reply.

In [5]:
conversation_history = []

def chat(user_message):
    conversation_history.append({"role": "user", "content": user_message})
    messages = [{"role": "system", "content": system_prompt}] + conversation_history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_responses = handle_tool_call(message)
        messages.append(message)
        messages.extend(tool_responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    reply = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": reply})
    return reply

## Step 4: Run the tracker

Defines `run_tracker()`, which displays a welcome message and loops over user input — calling `chat()` and rendering each reply as Markdown — until the user types "quit".

In [ ]:
def run_tracker():
    display(Markdown("""
# Accessibility Repo Tracker
Ask about open issues or the latest release for axe-core, eslint-plugin-jsx-a11y, or NVDA.
Type 'quit' to end the session.
"""))
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() == "quit":
            break
        if not user_input:
            continue
        reply = chat(user_input)
        display(Markdown("---"))
        display(Markdown(reply))

run_tracker()


# Accessibility Repo Tracker
Ask about open issues or the latest release for axe-core, eslint-plugin-jsx-a11y, or NVDA.
Type 'quit' to end the session.


---

Here's a summary of the open issues in **eslint-plugin-jsx-a11y** related to ARIA attributes:

---

### 🏷️ Labeled Issues

**`new-rule` / `question`**
- **[#986 — Rule Idea: Enforce boolean literals for "booleanish" HTML attributes such as `aria-hidden`](https://github.com/jsx-eslint/eslint-plugin-jsx-a11y/issues/986)**
  *(Opened: May 2024)*
  Proposes a new rule to enforce that booleanish ARIA attributes (like `aria-hidden`) use actual boolean literals (`true`/`false`) instead of string equivalents (`"true"`/`"false"`).

---

### 🔓 Unlabeled Issues

- **[#839 — Custom ARIA property support](https://github.com/jsx-eslint/eslint-plugin-jsx-a11y/issues/839)**
  *(Opened: March 2022)*
  Requests support for custom/non-standard ARIA properties (e.g., `aria-*` attributes not in the ARIA spec) without triggering false positives.

- **[#910 — `role-supports-aria-props` is not flagging prohibited accessible name usage on generic elements](https://github.com/jsx-eslint/eslint-plugin-jsx-a11y/issues/910)**
  *(Opened: December 2022)*
  Reports a bug where the `role-supports-aria-props` rule fails to flag disallowed ARIA properties used on generic/non-semantic elements.

- **[#946 — `jsx-a11y/iframe-has-title`: title should not be required for hidden frames](https://github.com/jsx-eslint/eslint-plugin-jsx-a11y/issues/946)**
  *(Opened: July 2023)*
  Suggests that `<iframe>` elements hidden via `aria-hidden` or similar should be exempt from the `iframe-has-title` rule.

---

### 💡 Key Takeaways
- There's an ongoing ask for **custom ARIA attribute support** (longstanding since 2022).
- A **bug exists** in `role-supports-aria-props` around generic elements.
- A **new rule proposal** for stricter boolean ARIA attribute enforcement is under discussion.

Would you like more details on any of these?